# Phase 2: LangGraph + CrewAI Mesh

## Architecture

The mesh extends Phase 1 with a CrewAI crew replacing the three sequential LangGraph nodes.

```
START → supervisor_node → crew_node → END
```

Inside `crew_node`, CrewAI runs two specialists sequentially:
1. **Triage Specialist** — reads feedback, writes `outputs/triage.csv`
2. **Brief and Response Specialist** — reads triage output, writes `outputs/brief.md` and `outputs/response_templates.md`

## Why this structure?
- LangGraph handles orchestration and state persistence
- CrewAI handles the agent-to-agent delegation within the crew
- This creates two layers of state: LangGraph state (measured at `crew_node` return) and CrewAI inter-agent messages (counted from `result.tasks_output`)

## Phase 2 measurements
These two fields are NOT in Phase 1 and are the primary comparison metrics:
- `message_count`: number of completed CrewAI tasks (proxy for inter-agent messages)
- `state_size_bytes`: serialised LangGraph state size at `crew_node` return

## How to run
See `01_run.ipynb` — load `config/mesh.yaml`, run sweep, inspect outputs.

## Handoff to Phase 3
Phase 3 interns inject failures into this mesh. See `src/aip_intern/failures/` stubs.


In [ ]:
from unittest.mock import MagicMock
from aip_intern.core.config import LLMCfg
from aip_intern.mesh.graph import build_graph

# Mock LLMCfg to draw the graph without a real LLM
class _MockLLMCfg:
    model = "openai/gpt-4o-mini"
    base_url = "http://mock"
    api_key = "mock"
    temperature = 0.0

graph = build_graph(_MockLLMCfg())
print(graph.get_graph().draw_mermaid())


## Key files

| File | Purpose |
|------|---------|
| `src/aip_intern/mesh/nodes.py` | supervisor_node |
| `src/aip_intern/mesh/crew_node.py` | crew_node async bridge |
| `src/aip_intern/mesh/graph.py` | Graph assembly |
| `src/aip_intern/mesh/runner.py` | Sweep loop |
| `config/mesh.yaml` | Run parameters |
| `scripts/run_mesh.py` | CLI entry point |

## Notes for Phase 3

The `message_count` field is currently a proxy (number of completed tasks).
Phase 3 interns: consider instrumenting CrewAI's callback hooks for finer-grained message counting.
